# 🔦 06 · Contrast Mechanisms: Bright-Field, Dark-Field, Phase Contrast, and DIC
### Making an invisible, transparent specimen visible without staining it

**Part B · Seeing like a microscope** · Module 06 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/06_contrast_mechanisms_bright_field_dark_field_phase_contrast_and_dic.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib |
| **Prerequisites** | Modules 03–05 |
| **Estimated time** | 75–90 minutes |
| **Instrumentation topics covered** | The weak-phase-object problem in plain bright-field; dark-field imaging by discarding the direct beam; Zernike phase contrast and its halo artifact; differential interference contrast (DIC) as a directional-derivative technique; choosing a contrast method for a given specimen and question |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every phase object, bright-field/dark-field/phase-contrast/DIC image here is a SIMULATED weak-phase-object rendering; no real stained or unstained specimen image is read from a file.

## 🎯 Learning objectives

1. Explain, using the weak-phase-object model, why a transparent, unstained specimen with
   essentially no absorption produces almost no contrast under plain bright-field illumination.
2. Simulate dark-field contrast by discarding the undiffracted (direct) beam, and explain why
   dark-field images appear as bright structures on an otherwise black background.
3. Simulate Zernike phase-contrast imaging, explain the physical role of the phase plate, and
   recognize the technique's characteristic halo artifact.
4. Simulate DIC imaging as a directional-derivative-sensitive technique, and explain both its
   characteristic pseudo-3-D "shadow-cast" appearance and its lack of a halo artifact.
5. Choose an appropriate contrast technique for a described specimen and biological question, and
   justify the choice against the technique's known trade-offs.

## 🗺️ Contents

1. The weak-phase-object problem: why bright-field makes transparent specimens disappear
2. Dark-field: discard the direct beam, keep only the scattered light
3. Zernike phase contrast: shift and attenuate the direct beam instead of blocking it
4. Differential interference contrast (DIC): interference between two sheared wavefronts
5. Choosing a contrast technique: a decision in practice

## 🧫 The biological question

A living, unstained neuron in culture is almost perfectly transparent under plain illumination — not
because the microscope is broken, but because the neuron barely absorbs any light at all. Nearly all
of the information about its shape is encoded in tiny delays it imposes on light passing through it
(its **phase**), not in how much light it soaks up (its **amplitude**). Staining the cell to make it
absorb light would usually kill it, which defeats the purpose of live-cell imaging entirely. Four
contrast techniques exist for exactly this reason — to turn invisible phase structure into visible
intensity structure, without ever touching the specimen's chemistry — and they trade off against each
other in ways that matter for the experiment, not just the picture.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
print('No GPU is needed anywhere in this notebook — everything is plain NumPy/Matplotlib.')

In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
SEED = 6
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.1; copied so this notebook runs alone) ──
import numpy as np

def make_phase_object(size: int = 140, seed: int = 0) -> np.ndarray:
    """A SIMULATED optical-path-length (phase) map of a thin, transparent 'cell': a smooth
    rotationally-symmetric body plus a few organelle-like bumps and dips of differing
    refractive index, in radians of phase shift relative to the surrounding medium."""
    rng = np.random.default_rng(seed)
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    cy, cx, r = size / 2, size / 2, size * 0.32
    phase = 0.5 * np.exp(-(((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * (r * 0.6) ** 2)))
    for _ in range(4):
        by, bx = rng.uniform(size * 0.3, size * 0.7, size=2)
        sign = rng.choice([-1.0, 1.0])
        br = rng.uniform(size * 0.04, size * 0.08)
        phase = phase + sign * 0.3 * np.exp(-(((yy - by) ** 2 + (xx - bx) ** 2) / (2 * br ** 2)))
    return phase


def zernike_phase_contrast(phase_map: np.ndarray, phase_shift: float = np.pi / 2,
                            attenuation: float = 0.25) -> np.ndarray:
    """Simulate Zernike phase-contrast imaging of a weak-phase object: the complex field
    exp(i * phase_map) is split into its mean ('undiffracted', zero-order) and zero-mean
    ('diffracted') parts; the phase plate advances the undiffracted part's phase by
    ``phase_shift`` and attenuates its amplitude, approximating the physical phase ring in the
    objective's back focal plane. Returns the resulting intensity image, including the
    technique's characteristic halo artifact."""
    field = np.exp(1j * phase_map)
    undiffracted = field.mean()
    diffracted = field - undiffracted
    shifted = undiffracted * attenuation * np.exp(1j * phase_shift)
    out_field = shifted + diffracted
    return np.abs(out_field) ** 2


def dic_shear_render(phase_map: np.ndarray, shear_px: int = 2, bias: float = 0.4) -> np.ndarray:
    """Simulate differential interference contrast (DIC): two laterally-sheared copies of the
    same wavefront interfere, so contrast is proportional to the directional derivative (here,
    along the sheared axis) of the optical path length, giving the technique's characteristic
    pseudo-3-D 'shadow-cast' relief. ``bias`` is the Nomarski prism's bias retardation."""
    sheared = np.roll(phase_map, shear_px, axis=1)
    path_diff = phase_map - sheared
    return 0.5 * (1 + np.cos(bias + path_diff))

---
## 1 · The weak-phase-object problem: why bright-field makes transparent specimens disappear

A thin, transparent specimen barely absorbs light, but it does slow it down slightly wherever
its refractive index differs from the surrounding medium — thicker or denser regions delay the
wavefront a little more than their surroundings. This delay is captured as a **phase map**
$\phi(x,y)$ (in radians), and the field just after the specimen is $\exp(i\phi(x,y))$ — a complex
number of *constant magnitude 1* everywhere, varying only in phase. Plain bright-field intensity is
simply $|\text{field}|^2$. Because the magnitude never varies, that intensity comes out essentially
flat, **regardless of how much structure is in $\phi$**: a camera recording plain transmitted light
cannot tell a fully-formed organelle from empty medium, because neither one absorbs any light.

In [ ]:
# ── 1.1 a simulated phase object is invisible in plain bright-field ──
phase_map = make_phase_object(size=160, seed=SEED)
brightfield = np.abs(np.exp(1j * phase_map)) ** 2

print(f'Phase map ranges from {phase_map.min():.3f} to {phase_map.max():.3f} radians (real structure).')
print(f'Bright-field intensity std. dev. across the whole image: {brightfield.std():.2e} (essentially flat).')

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
im0 = axes[0].imshow(phase_map, cmap='viridis')
axes[0].set_title('Ground-truth phase map\n(the specimen\'s real structure)')
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(brightfield, cmap='gray', vmin=0, vmax=2)
axes[1].set_title('Plain bright-field intensity\n(what a camera actually records)')
plt.colorbar(im1, ax=axes[1], fraction=0.046)
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The phase map on the left has clear, real structure — a cell body and
several organelle-like features — yet the bright-field intensity on the right is, to within numerical
precision, perfectly flat. This is the textbook motivation for every technique in this module: Zernike
himself developed phase contrast specifically to solve exactly this problem, which is why the
technique bears his name. (In practice, bright-field of a real specimen is never *perfectly* flat —
slight defocus and partial coherence effects can produce faint edge contrast, called Becke lines —
but the dominant effect for a well-focused, weakly-scattering specimen is exactly this near-total
invisibility.)

✏️ **Your turn.** Multiply `phase_map` by `3.0` before computing `brightfield` (a much "thicker"
simulated specimen). Does the bright-field intensity std. dev. increase meaningfully? At what phase
magnitude does the weak-phase-object approximation clearly start to break down?

---
## 2 · Dark-field: discard the direct beam, keep only the scattered light

The field transmitted through the specimen can be split into two parts: an **undiffracted**
(direct) component — simply the field's average value, representing light that passed straight
through as if the specimen were not there — and a **diffracted** (scattered) component, the
zero-mean remainder that carries all of the specimen's actual structure. **Dark-field** microscopy
places a physical stop that blocks only the direct beam, so only the diffracted light ever reaches
the detector. Structures appear as bright scattered light against a background that is genuinely
black, because there is no direct beam left to form a background at all.

In [ ]:
# ── 2.1 dark-field: keep only the diffracted (zero-mean) component ──
field = np.exp(1j * phase_map)
undiffracted = field.mean()
diffracted = field - undiffracted
dark_field = np.abs(diffracted) ** 2

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
axes[0].imshow(brightfield, cmap='gray', vmin=0, vmax=2); axes[0].set_title('Bright-field\n(flat, uninformative)')
axes[1].imshow(dark_field, cmap='gray'); axes[1].set_title('Dark-field\n(structure on a black background)')
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The same phase map that was invisible in Section 1 now produces
clearly visible bright structure against a black field. Dark-field is extremely sensitive to small
scatterers (it is often used to see sub-resolution particles that scatter enough light to detect, even
though diffraction prevents resolving their true shape — Module 04's PSF framework still applies to
*where* that scattered light lands), but it does not straightforwardly report the specimen's actual
phase value at each point — it reports *how much the local phase deviates from the average*, which is
a related but different quantity.

✏️ **Your turn.** Compare the brightest pixel's value in `dark_field` against `brightfield`'s
background intensity (around `1.0`). Dark-field's absolute intensity scale has a different meaning
than bright-field's — why does comparing the two numbers directly not make much physical sense?

---
## 3 · Zernike phase contrast: shift and attenuate the direct beam instead of blocking it

Dark-field throws the direct beam away entirely. Zernike's insight was that interfering the
direct beam with the diffracted light — rather than discarding it — converts invisible phase
differences directly into visible intensity differences, through ordinary wave interference. A
physical **phase plate** in the objective's back focal plane advances the undiffracted beam's phase
by about a quarter wave ($\pi/2$) and attenuates its amplitude; `zernike_phase_contrast` implements
exactly this as a complex-number operation on the same field decomposition from Section 2.

In [ ]:
# ── 3.1 Zernike phase contrast, and its characteristic halo artifact ──
phase_contrast_image = zernike_phase_contrast(phase_map, phase_shift=np.pi / 2, attenuation=0.25)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
axes[0].imshow(phase_map, cmap='viridis'); axes[0].set_title('Ground-truth phase map')
axes[1].imshow(phase_contrast_image, cmap='gray'); axes[1].set_title('Zernike phase contrast')
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

# line profile across the main cell body, to see the halo directly
row = 80
profile_phase = phase_map[row, :]
profile_pc = phase_contrast_image[row, :]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax2 = ax.twinx()
ax.plot(profile_phase, color='teal', label='ground-truth phase')
ax2.plot(profile_pc, color='crimson', label='phase-contrast intensity')
ax.set_ylabel('phase (radians)', color='teal'); ax2.set_ylabel('intensity', color='crimson')
ax.set_xlabel('x (pixels)'); ax.set_title('A line profile through the cell body: note the bright/dark halo at its edge')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The cell body and its organelles are now clearly visible as intensity
variations, unlike the flat bright-field image — but look closely at the line profile: right at the
edge of the simulated cell body, the phase-contrast intensity overshoots *past* the surrounding
background level before settling, producing a bright or dark fringe around the structure that is not
actually present in the ground-truth phase map. This is the **halo artifact**, an unavoidable
consequence of attenuating (rather than discarding) the undiffracted beam, and it is the single most
common reason phase-contrast images are considered unreliable for quantitative intensity
measurements right at object boundaries.

✏️ **Your turn.** Change `attenuation` to `0.1` (a stronger phase plate) and re-run. Does the halo
get more or less pronounced? What does this suggest about why the standard `attenuation` design
value is a specific, carefully chosen compromise rather than "as strong as possible"?

### 📘 Lecture notes — the halo is the price phase contrast pays for working without a stop

**Definition.** The phase-contrast halo is a bright or dark fringe surrounding specimen features,
caused by attenuating and phase-shifting (rather than fully blocking) the undiffracted beam —
visible directly in Section 3's line profile as an overshoot that is not present in the true phase
map.

**Worked example.** A thick, strongly-phase-shifting region (like a dense nucleolus) produces a much
more pronounced halo than a weakly-phase-shifting one, which is why phase contrast is best suited to
thin, genuinely weak-phase specimens — exactly the regime this module's `make_phase_object` is built
to represent, and progressively less reliable outside it.

**One memorable point.** *Phase contrast does not measure phase directly — it converts phase into
intensity through an artifact-prone shortcut; DIC and quantitative phase imaging (a later module)
take different routes for exactly this reason.*

---
## 4 · Differential interference contrast (DIC): interference between two sheared wavefronts

Rather than splitting the beam into undiffracted/diffracted components, **DIC** splits it into
two laterally-sheared copies of the *same* wavefront (via a Nomarski prism), displaced from each
other by a tiny distance, which are then recombined and made to interfere. The resulting intensity
depends on the *difference* between the phase map and a shifted copy of itself — a numerical
derivative along the shear direction — which is exactly what `dic_shear_render` computes. Because DIC
responds to the *rate of change* of phase rather than its absolute value, flat regions (even with
large absolute phase) stay featureless, while edges — wherever the phase changes quickly — produce
strong, directional contrast.

In [ ]:
# ── 4.1 DIC contrast is directional: build a specimen with features at several orientations ──
size = 160
yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')

def ridge(center_y, center_x, angle_deg, length, width, amplitude):
    theta = np.radians(angle_deg)
    along = (yy - center_y) * np.sin(theta) + (xx - center_x) * np.cos(theta)
    across = -(yy - center_y) * np.cos(theta) + (xx - center_x) * np.sin(theta)
    return amplitude * np.exp(-(along ** 2) / (2 * (length / 2) ** 2) - (across ** 2) / (2 * (width / 2) ** 2))

bars_phase = (ridge(50, 80, 0, 90, 10, 0.9)      # horizontal bar: phase varies mostly along y
              + ridge(110, 80, 90, 90, 10, 0.9)   # vertical bar: phase varies mostly along x
              + ridge(80, 40, 45, 70, 10, 0.9))    # diagonal bar

dic_image = dic_shear_render(bars_phase, shear_px=3, bias=0.4)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
axes[0].imshow(bars_phase, cmap='viridis'); axes[0].set_title('Ground-truth phase: three bars,\ndifferent orientations')
axes[1].imshow(dic_image, cmap='gray'); axes[1].set_title('DIC (shear along x): only some\nedges show strong contrast')
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** `dic_shear_render` shears along the x axis (columns), so it is most
sensitive to phase changing *along x* — which is exactly the direction the vertical bar's edges cut
across. The horizontal bar's edges, running mostly along the shear direction itself, produce visibly
weaker contrast. This directionality is a genuine, well-known limitation of real DIC: a specimen
feature aligned just right relative to the shear axis can become nearly invisible, which is why DIC
microscopes let the operator physically rotate the prism assembly (or the specimen) to find the best
orientation. Note also that DIC's shadow-cast relief has no halo artifact comparable to Section 3's —
a genuine advantage for quantitative boundary measurements.

✏️ **Your turn.** Change `dic_shear_render`'s `shear_px` from `3` to `1` and then to `8`. How does
the strength and sharpness of the shadow-cast relief change with shear distance? (Real DIC prisms are
manufactured with a fixed shear distance for exactly this reason — it is a hardware design choice, not
a software setting.)

### 📘 Lecture notes — DIC trades a halo for a direction

**Definition.** DIC contrast is proportional to the phase map's directional derivative along the
shear axis, producing a pseudo-3-D relief appearance with no halo artifact, but with contrast that
depends on feature orientation relative to the shear direction.

**Worked example.** A radially symmetric structure (like the circular cell body from Sections 1-3)
renders differently under DIC than under phase contrast: one side of the structure appears raised
and the opposite side appears shadowed, rather than the uniform halo-ringed disc phase contrast would
produce — try it directly by passing `phase_map` from Section 1 into `dic_shear_render`.

**One memorable point.** *If phase contrast's weakness is a halo around every edge, DIC's weakness is
blindness to edges running the "wrong" way — neither artifact is fixable by better specimen
preparation, because both are consequences of the optical design itself.*

---
## 5 · Choosing a contrast technique: a decision in practice

No single technique in this module is strictly "better" — each is suited to a different
combination of specimen and question:

| Technique | Visible without staining? | Halo artifact | Directional blind spot | Compatible with simultaneous fluorescence? | Typical use |
|---|---|---|---|---|---|
| Bright-field | No (weak-phase specimens) | — | — | Yes | Stained/absorbing specimens, quick checks |
| Dark-field | Yes | No | No | Limited (scattered light overwhelms weak fluorescence) | Sub-resolution particles, edges, unstained live cells |
| Phase contrast | Yes | Yes | No | Yes (common pairing with fluorescence channels) | Routine live-cell phase imaging, thin monolayers |
| DIC | Yes | No | Yes | Yes (common pairing with fluorescence channels) | High-contrast live-cell detail, thicker/3-D-looking specimens |

The choice is rarely about which image "looks nicest" — it is about which artifact the downstream
analysis can tolerate least. A segmentation pipeline that is sensitive to boundary intensity would
rather deal with DIC's directional blind spot (often fixable by choosing the specimen's or the
prism's orientation) than phase contrast's halo (which directly corrupts the boundary measurement
itself, from any orientation).

✏️ **Your turn.** A core question in this course's segmentation modules is finding accurate cell
boundaries from image intensity alone. Based on this module's artifacts, which single technique from
the table would you expect to be hardest to segment reliably with a simple threshold on raw
intensity, and why?

## 📌 Key takeaways

- A transparent, unstained specimen's structure lives in its phase, not its amplitude; plain
  bright-field intensity is nearly blind to phase alone.
- Dark-field discards the direct beam entirely, rendering scattered structure as bright features on a
  black background, at the cost of a less directly interpretable intensity scale.
- Zernike phase contrast interferes a phase-shifted, attenuated direct beam with the diffracted light,
  converting phase into intensity at the cost of a halo artifact around every structure's edges.
- DIC interferes two laterally-sheared copies of the wavefront, producing a halo-free, directionally
  sensitive pseudo-3-D relief image.
- Choosing a contrast technique is a trade-off decision driven by what the downstream analysis needs
  to tolerate, not a search for a single universally "best" method.

## ✅ Checkpoint

**1.** A specimen that absorbs essentially no light produces an almost perfectly flat bright-field
image. Is the microscope malfunctioning?

<details><summary>Answer</summary>
No. As Section 1 showed, this is the expected, physically correct result for a weak-phase object —
$|\exp(i\phi)|^2 = 1$ everywhere, regardless of how much real structure is encoded in $\phi$. The
specimen's structure is there; plain bright-field intensity simply cannot see it.
</details>

**2.** An image shows a bright or dark fringe running around the outside of every cell, which is not
present in independently-measured ground truth. Which contrast technique in this module is most
likely responsible, and what is this artifact called?

<details><summary>Answer</summary>
Zernike phase contrast; the artifact is the halo, caused by attenuating and phase-shifting (rather
than fully removing) the undiffracted beam, as demonstrated directly in Section 3's line profile.
</details>

**3.** A specimen's long, thin filamentous structures are clearly visible in one DIC image but nearly
invisible in a second DIC image of the same specimen rotated 90°. Is this evidence that the specimen
changed between images?

<details><summary>Answer</summary>
No. Section 4 showed that DIC contrast depends on the phase gradient's component along the shear
axis; a filament oriented along the shear direction produces much weaker contrast than one oriented
across it. Rotating the specimen (or the prism) changes which features align with the shear axis, not
the specimen's actual structure.
</details>

## 🧠 Questions & discussion

- Quantitative phase imaging (QPI) techniques, covered in a later part of this course, aim to recover
  the actual phase map $\phi(x,y)$ itself, rather than an intensity image derived from it. Based on
  this module's artifacts, why might a biologist want the actual phase value at each pixel rather
  than any of the four contrast images shown here?
- Dark-field is described here as having "a less directly interpretable intensity scale." What
  additional piece of information would you need to convert a dark-field intensity value into
  something quantitatively meaningful about the specimen?
- Phase contrast and DIC are both commonly used alongside fluorescence channels in the same
  experiment. Why might dark-field be a comparatively poor choice to combine with a dim fluorescent
  reporter, even though it works perfectly well on its own?

## 🚀 Bonus — going further

- Pass `phase_map` from Section 1 (the radially symmetric cell body) into `dic_shear_render` instead
  of the oriented bars from Section 4, and describe the resulting "shadow-cast" appearance in your
  own words.
- Try combining two contrast mechanisms conceptually: what would a phase-contrast image look like if
  computed from only the *dark-field* diffracted component, skipping the undiffracted term entirely?
  (Hint: look again at the decomposition in Section 2 versus Section 3's phase-plate formula.)
- Vary `zernike_phase_contrast`'s `phase_shift` continuously from `0` to `np.pi` and observe how the
  image transitions between dark-contrast and bright-contrast conventions — this is exactly the
  physical effect of swapping a "positive" for a "negative" phase plate on a real microscope.

## 📝 Lab note

```text
Module: 06 — Contrast Mechanisms (Bright-Field, Dark-Field, Phase Contrast, DIC)
Date:
The contrast technique I would choose for MY own specimen and question, and why:
One artifact from this module I would now watch for before trusting a boundary measurement:
```

## 📚 References

1. Zernike, F. (1955). How I discovered phase contrast. *Science*, 121(3141), 345-349.
2. Allen, R.D., David, G.B., Nomarski, G. (1969). The Zeiss-Nomarski differential interference
   equipment for transmitted-light microscopy. *Zeitschrift fur wissenschaftliche Mikroskopie*,
   69(4), 193-221.
3. Murphy, D.B., Davidson, M.W. (2012). *Fundamentals of Light Microscopy and Electronic Imaging*
   (2nd ed.). Wiley-Blackwell. — a fuller treatment of all four contrast techniques and their optics.
4. Inoue, S., Spring, K.R. (1997). *Video Microscopy: The Fundamentals* (2nd ed.). Springer.

## ➡️ Next

Module 07 turns to fluorescence microscopy instrumentation — a fundamentally different contrast
mechanism that does not rely on phase or scattering at all, but on specific molecules emitting their
own light, and the optical hardware (light sources, filter cubes, dichroic mirrors) built specifically
to isolate that emitted light from everything else.